# 06 · Avaliação — qual modelo escolher e como ele se sai num mês novo
Lê os onze algoritmos treinados no `05_treino` direto do MLflow, compara com uma referência simples, escolhe o modelo e avalia no out-of-time: curva ROC, calibração das probabilidades, precisão no top 50 e importância das features por bloco.

**Entrada:** experimento `churn-teo-me-why` (MLflow) e `abt_churn_bases_v0` · **Saída:** `output/figs/12_*` a `14_*` e `output/tables/comparacao_modelos_v0.csv`

In [0]:
%pip install feature_engine catboost

In [0]:
%run ./00_setup

Resultado do Modelo

In [0]:
import os
os.environ['MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR'] = 'false'   # esconde as barras de download dos modelos

import mlflow
from sklearn import metrics, pipeline

VERSAO      = 'v0'                                   # mesma versão do 05_treino
EXPERIMENTO = f'/Users/{USUARIO}/churn-teo-me-why'    # experimento do MLflow

# bases salvas pelo 05_treino (seção 1.5): nada é sorteado ou treinado de novo
bases = spark.table(f'{TABELA_ABT}_bases_{VERSAO}').toPandas()
features = bases.columns.drop(['IdCliente', 'dtRef', 'churn', 'base']).tolist()

treino = bases[bases['base'] == 'treino']
teste  = bases[bases['base'] == 'teste']
df_oot = bases[bases['base'] == 'oot']

X_train, y_train = treino[features], treino['churn']
X_test,  y_test  = teste[features],  teste['churn']
X_oot,   y_oot   = df_oot[features], df_oot['churn']

## 1. Referência de comparação

In [0]:
acc_todos_saem = metrics.accuracy_score(y_test, [1] * len(y_test))   # churn é a classe majoritária (72%)
auc_recencia   = metrics.roc_auc_score(y_test, X_test['recencia'])
print(f'Chutar "todo mundo sai": acurácia {acc_todos_saem:.3f}')
print(f'Recência sozinha: AUC {auc_recencia:.3f}')

## 2. Comparação dos algoritmos
Métricas dos onze runs, lidas do experimento do MLflow. A escolha usa a AUC da validação cruzada: com onze modelos, escolher pelo teste favorece quem acertou por acaso.

In [0]:
runs = mlflow.search_runs(
    experiment_names=[EXPERIMENTO],
    filter_string="attributes.status = 'FINISHED'",
    order_by=['start_time DESC'],
)
runs = runs.dropna(subset=['metrics.auc_cv', 'metrics.auc_test'])            # só runs com as métricas
if 'tags.versao' in runs.columns and (runs['tags.versao'] == VERSAO).any():  # só a versão atual, se marcada
    runs = runs[runs['tags.versao'] == VERSAO]
runs = runs.drop_duplicates(subset='tags.mlflow.runName', keep='first')     # o mais recente de cada algoritmo

comparacao = (
    runs.rename(columns={
        'tags.mlflow.runName': 'modelo',
        'metrics.auc_train':   'auc_treino',
        'metrics.auc_cv':      'auc_cv',
        'metrics.auc_test':    'auc_teste',
    })
    .set_index('modelo')[['auc_treino', 'auc_cv', 'auc_teste', 'run_id']]
)
comparacao['diferenca'] = comparacao['auc_treino'] - comparacao['auc_teste']
comparacao.loc['referência: recência', 'auc_teste'] = auc_recencia
comparacao = comparacao.sort_values('auc_cv', ascending=False)

comparacao.drop(columns='run_id').to_csv(f'{TABLE_DIR}/comparacao_modelos_{VERSAO}.csv')
comparacao.round(3)

## 3. Escolha do modelo
Random Forest e CatBoost empataram em primeiro (AUC cv 0,806). O Random Forest foi escolhido: mesmo desempenho, mais simples, sem dependência extra e com a importância das features pronta.

In [0]:
MELHOR = comparacao.drop("referência: recência")["auc_cv"].idxmax()
# ou fixe: MELHOR = "random_forest"   # empatado com o catboost; mais simples e sem dependência extra

modelo_final = mlflow.sklearn.load_model(f"runs:/{comparacao.loc[MELHOR, 'run_id']}/model")
print(f"Modelo escolhido: {MELHOR}")

### 3.1 Curva ROC dos 4 melhores no teste
Cada curva mostra, para cada ponto de corte, quanto o modelo acerta de quem sai (eixo vertical) contra quantos alarmes falsos ele dá (eixo horizontal). Quanto mais perto do canto superior esquerdo, melhor.

In [0]:
top4 = comparacao.drop('referência: recência').head(4).index

fig, ax = plt.subplots(figsize=(7, 6))
for nome_modelo, cor in zip(top4, BLUE_PALETTE):
    modelo_run = mlflow.sklearn.load_model(f"runs:/{comparacao.loc[nome_modelo, 'run_id']}/model")
    prob = modelo_run.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = metrics.roc_curve(y_test, prob)
    ax.plot(fpr, tpr, color=cor, lw=2, label=f'{nome_modelo} (AUC {metrics.roc_auc_score(y_test, prob):.3f})')

fpr, tpr, _ = metrics.roc_curve(y_test, X_test['recencia'])
ax.plot(fpr, tpr, color=COR_NEUTRA, lw=1.5, ls='--', label=f'recência sozinha (AUC {auc_recencia:.3f})')
ax.plot([0, 1], [0, 1], color=COR_NEUTRA, lw=1, ls=':', label='chute (AUC 0,5)')

ax.set(title='Curva ROC no teste: os melhores modelos praticamente empatam',
       xlabel='taxa de falsos positivos (quem ficou, mas o modelo disse que sai)',
       ylabel='taxa de verdadeiros positivos (quem saiu e o modelo acertou)')
ax.legend(loc='lower right')
salvar(fig, '12_roc_comparacao_modelos')

## 4. Assess: desempenho no out-of-time
O teste definitivo: um mês (ago/26) que o modelo nunca viu, com todos os usuários ativos, e não uma safra sorteada por usuário.

### 4.1 Curva ROC em treino, teste e out-of-time
Se as curvas ficarem próximas, o modelo não decorou o treino e se mantém no tempo.

In [0]:
fig, ax = plt.subplots(figsize=(7, 6))
for nome_base, X_base, y_base, cor in [
    ('treino',      X_train, y_train, BLUE_PALETTE[0]),
    ('teste',       X_test,  y_test,  BLUE_PALETTE[1]),
    ('out of time', X_oot,   y_oot,   BLUE_PALETTE[2]),
]:
    prob = modelo_final.predict_proba(X_base)[:, 1]
    fpr, tpr, _ = metrics.roc_curve(y_base, prob)
    ax.plot(fpr, tpr, color=cor, lw=2, label=f'{nome_base} (AUC {metrics.roc_auc_score(y_base, prob):.3f})')

ax.plot([0, 1], [0, 1], color=COR_NEUTRA, lw=1, ls=':', label='chute (AUC 0,5)')
ax.set(title=f'Curva ROC do modelo escolhido ({MELHOR})',
       xlabel='taxa de falsos positivos', ylabel='taxa de verdadeiros positivos')
ax.legend(loc='lower right')
salvar(fig, '13_roc_modelo_final')

### 4.2 Calibração das probabilidades
A probabilidade prevista bate com o churn real? O treino tem 72% de churn (uma safra por usuário) e o out-of-time, 44%. Se o modelo herdou a taxa do treino, as probabilidades saem altas demais.

In [0]:
prob_oot = modelo_final.predict_proba(X_oot)[:, 1]
print(f'Probabilidade média prevista: {prob_oot.mean():.1%} | churn real no out-of-time: {y_oot.mean():.1%}')

# por faixa de probabilidade: prevista × real
calib = (pd.DataFrame({'prob': prob_oot, 'real': y_oot.values})
           .assign(faixa=lambda d: pd.qcut(d['prob'], 5, duplicates='drop'))
           .groupby('faixa', observed=True)
           .agg(usuarios=('real', 'size'), prob_media=('prob', 'mean'), churn_real=('real', 'mean')))
calib.round(3)

### 4.3 Precisão no top 50
Dos 50 usuários com maior probabilidade de churn no out-of-time, quantos realmente saíram. É a métrica que responde à pergunta de negócio.

In [0]:
ranking_oot = (df_oot[['IdCliente', 'churn']]
               .assign(prob_churn=prob_oot)
               .sort_values('prob_churn', ascending=False))

precisao_top50 = ranking_oot.head(50)['churn'].mean()
print(f'Precisão no top 50: {precisao_top50:.0%} | churn médio do out-of-time: {y_oot.mean():.0%} '
      f'| ganho: {precisao_top50 / y_oot.mean():.1f}x')

## 5. Importância das features por bloco
Quais blocos do MECE o modelo mais usa para prever o churn.

In [0]:
BLOCOS = {   # C vem primeiro: diasAtivosD28Anterior também contém 'diasAtivos'
    'C. Tendência':  ['diasAtivosD28Anterior', 'deltaDiasD28', 'tendencia_'],
    'A. Perfil':     ['diasDesdePrimeira', 'flNovatoRisco', 'qtdRedes'],
    'B. Volume':     ['recencia', 'diasAtivos', 'qtdTransacoesD28', 'pontosGanhosD28', 'freqVida',
                      'transacoesPorDiaD28', 'horasPorDiaD28', 'percentilPontos', 'zScorePontos'],
    'D. Mix':        ['share_', 'produtoDominante', 'qtdProdutosDistintos'],
    'E. Hábito':     ['qtdMadrugada', 'qtdManha', 'qtdTarde', 'qtdNoite', 'qtdDiasSemana', 'periodoDominante',
                      'Intervalo', 'qtdRetornos', 'razaoAtraso', 'fezStreak', 'diasUltimoStreak'],
    'F. Comunidade': ['audienciaMedia', 'pctLivesPresente', 'livesPerdidas'],
    'G. Loja':       ['pontosGastosD28', 'saldoPontos', 'gastouD28'],
    'H. Educação':   ['Curso', 'pctMedioCompleto', 'qtdEpsD28', 'diasDesdeUltimoEp'],
}
def bloco(col):
    for b_, padroes in BLOCOS.items():
        if any(p in col for p in padroes):
            return b_
    return 'outros'

clf_final = modelo_final[-1]
if isinstance(clf_final, pipeline.Pipeline):   # modelos com escala: o classificador é o último passo
    clf_final = clf_final[-1]
colunas = modelo_final[:-1].transform(X_train.head()).columns

if hasattr(clf_final, 'feature_importances_'):
    peso = clf_final.feature_importances_
else:
    peso = np.abs(clf_final.coef_[0])

importancia = pd.DataFrame({'feature': colunas, 'importancia': peso / peso.sum()})
importancia['bloco'] = importancia['feature'].map(bloco)
por_bloco = importancia.groupby('bloco')['importancia'].sum().sort_values()

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(por_bloco.index, por_bloco * 100, color=COLOR_BLUE)
ax.bar_label(ax.containers[0], labels=[f'{v:.0%}' for v in por_bloco], padding=3, color=COLOR_TITLE)
ax.set(title=f'Importância das features por bloco ({MELHOR})', xlabel='% da importância total', ylabel='')
salvar(fig, '14_importancia_por_bloco')

importancia.sort_values('importancia', ascending=False).head(15).round(3)